# 05 – Foundation models
Zero-shot forecasts of the 24 hourly prices of day D with pre-trained foundation models, without fine-tuning and with default parameters (CLAUDE.md section 6):

| Model | Inputs |
|---|---|
| `timesfm` | last 2,048 h of prices |
| `timesfm_cov` | same + known-future covariates `load_fc`, `wind_fc`, `solar_fc`, `non_working_day` over context + D |
| `timestral` | last 2,048 h of prices |
| `timestral_cov` | same + the same known-future covariates |

The covariates are the same as SARIMAX, so the two-way table compares families with the same information. `hour` and `dow` are not passed: the models learn the daily and weekly cycles from the context. Covariates are always day-ahead **forecasts**, in the context as in the future.

Each model is run with the common rolling loop (`src.evaluation.run_rolling`) on the main test period and on the control period (contamination test, section 7). Forecasts are cached in `results/` and `results/control/`.

In [ ]:
import resource
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src import config as cfg
from src.data import load_dataset, split_for_day
from src.evaluation import mae, pinball, rmse, run_rolling
from src.foundation import Timestral, TimesFM3, get_device

df = load_dataset()
device = get_device()
CONTROL_DAYS = pd.date_range(cfg.CONTROL_START, cfg.CONTROL_END, freq="D")

print(f"Dataset: {df.index[0]} → {df.index[-1]}")
print(f"Main test: {len(cfg.TEST_DAYS)} days, control: {len(CONTROL_DAYS)} days ({CONTROL_DAYS[0].date()} → {CONTROL_DAYS[-1].date()})")
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name()})" if device == "cuda" else ""))

In [ ]:
def timed_run(model, name, days, results_dir=cfg.RESULTS_DIR):
    """run_rolling, plus wall-clock time and peak memory when the forecasts are actually computed."""
    cached = (results_dir / f"{name}.parquet").exists()
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    results = run_rolling(model, name, df, days, results_dir=results_dir)
    if not cached:
        elapsed = time.perf_counter() - start
        peak = (f"peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.2f} GB" if device == "cuda"
                else f"peak process RSS {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.2f} GB (CPU run)")
        print(f"  {elapsed / len(days) * 1000:.0f} ms per day, {peak}")
    return results

## 1. TimesFM 3

Google Research, released 31 August 2026, 331M parameters, weights `google/timesfm-3.0-pytorch` (non-commercial licence, fine for this academic project).

- `TimesFM3Forecaster` is used, **not** `TimesFM3Evaluator`: the evaluator defaults to `make_positive=True` (clips at 0 when the context is non-negative), which would forbid negative prices. The forecaster keeps `make_positive=False` and `use_symmetric_averaging=False`.
- `return_quantiles=True`: the model returns the 9 deciles 0.1 … 0.9, exactly the levels used for the pinball loss. The point forecast is the median (quantile 0.5).
- Inference is deterministic (no sampling), so no seed is needed.

In [ ]:
start = time.perf_counter()
timesfm_model = TimesFM3.load(device)
print(f"Loaded in {time.perf_counter() - start:.1f} s on {timesfm_model.device}, quantiles {timesfm_model.config.quantiles}")

timesfm = TimesFM3(timesfm_model, with_covariates=False)
timesfm_cov = TimesFM3(timesfm_model, with_covariates=True)

### 1.1 Check on one day

First day of the main test: context (last week shown), median and 10–90 % band of both versions, actual prices of D.

In [ ]:
day = cfg.TEST_DAYS[0]
history, future_cov = split_for_day(df, day)
actual = df.loc[future_cov.index, cfg.TARGET]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(history.index[-7 * 24:], history[cfg.TARGET].iloc[-7 * 24:], color="k", lw=1, label="context (last week)")
ax.plot(actual.index, actual, color="k", lw=1, ls="--", label="actual prices of D")
for model, label, color in [(timesfm, "TimesFM 3", "C0"), (timesfm_cov, "TimesFM 3 + cov.", "C1")]:
    y_pred, q = model.predict(history, future_cov)
    ax.plot(future_cov.index, y_pred, color=color, label=f"{label} median")
    ax.fill_between(future_cov.index, q["q10"], q["q90"], color=color, alpha=0.2, label=f"{label} 10–90 %")
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("EUR/MWh")
ax.set_title(f"TimesFM 3 forecasts for {day.date()}")
ax.legend(fontsize=8, ncol=3)
fig.tight_layout()
plt.show()

### 1.2 Rolling evaluation

One forecast per day D, moving forward one day at a time. Each call only receives `split_for_day(df, D)` (prices up to D−1 23:00, covariate forecasts and calendar of D).

In [ ]:
res = {}
for name, model in [("timesfm", timesfm), ("timesfm_cov", timesfm_cov)]:
    res[name] = timed_run(model, name, cfg.TEST_DAYS)

In [ ]:
res_control = {}
for name, model in [("timesfm", timesfm), ("timesfm_cov", timesfm_cov)]:
    res_control[name] = timed_run(model, name, CONTROL_DAYS, results_dir=cfg.RESULTS_DIR / "control")

### 1.3 First look at the results

MAE, RMSE and pinball loss per period, in EUR/MWh. rMAE, Diebold-Mariano tests and the comparison with the other models are in `06_results.ipynb`.

In [ ]:
def summary(results: dict) -> pd.DataFrame:
    return pd.DataFrame({
        name: {
            "MAE": mae(r["y_true"], r["y_pred"]),
            "RMSE": rmse(r["y_true"], r["y_pred"]),
            "pinball": pinball(r["y_true"], r[cfg.QUANTILE_COLUMNS]),
            "coverage 10–90 %": ((r["y_true"] >= r["q10"]) & (r["y_true"] <= r["q90"])).mean(),
            "negative forecasts": (r["y_pred"] < 0).sum(),
            "negative actuals": (r["y_true"] < 0).sum(),
        }
        for name, r in results.items()
    }).T

pd.concat({"main test": summary(res), "control": summary(res_control)}).round(3)

The forecaster does not clip: negative forecasts are possible and appear on the hours where the context and the covariates point to a solar surplus.

The 10–90 % band should contain about 80 % of the actual prices if the quantiles are calibrated.

In [ ]:
week = cfg.TEST_DAYS[:7]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, sharey=True)
for ax, (name, label) in zip(axes, [("timesfm", "TimesFM 3"), ("timesfm_cov", "TimesFM 3 + cov.")]):
    r = res[name][res[name]["target_date"].isin(week)]
    t = r["target_date"] + pd.to_timedelta(r["hour"], unit="h")
    ax.plot(t, r["y_true"], color="k", lw=1, label="actual")
    ax.plot(t, r["y_pred"], color="C0", lw=1, label="median")
    ax.fill_between(t, r["q10"], r["q90"], color="C0", alpha=0.2, label="10–90 %")
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_title(label)
    ax.set_ylabel("EUR/MWh")
    ax.legend(fontsize=8, loc="upper right")
fig.suptitle(f"Day-ahead forecasts, first week of the main test ({week[0].date()} → {week[-1].date()})")
fig.tight_layout()
plt.show()

In [ ]:
# MAE by hour of day, main test
by_hour = pd.DataFrame({
    name: r.assign(err=(r["y_true"] - r["y_pred"]).abs()).groupby("hour")["err"].mean() for name, r in res.items()
})
ax = by_hour.plot(figsize=(10, 3.5), marker="o", ms=3)
ax.set_ylabel("MAE [EUR/MWh]")
ax.set_title("TimesFM 3: MAE by hour of day, main test")
plt.tight_layout()
plt.show()

### 1.4 Compute

Run of 2026-11-19 on the NVIDIA RTX 3070 (8 GB, driver 580, PyTorch 2.15), 207 days per model (main test + control):

| Model | Time per day | Main test (183 days) | Control (24 days) | Peak VRAM |
|---|---|---|---|---|
| `timesfm` | ≈ 0.18 s | 34 s | 4 s | 1.24 GB |
| `timesfm_cov` | ≈ 0.19 s | 35 s | 4.5 s | 1.26 GB |

Model loading: ≈ 6 s (weights already in the Hugging Face cache). The time per day includes building the inputs and the results; the forecast itself matches the smoke test (≈ 155 ms, ≈ 1.3 GB peak VRAM).

The same notebook also runs on CPU (machine without an NVIDIA GPU, 12 threads): ≈ 0.24 s per day univariate and ≈ 0.64 s with covariates, about 3 minutes in total. CPU and GPU forecasts agree to within 0.001 EUR/MWh (floating-point differences only), so the results do not depend on the machine.


## 2. Timestral

Mistral AI, released 15 October 2026, largest size (209M parameters), weights `mistralai/Timestral-209M` (modified MIT licence), inference code `mistralai/timestral` 1.0.0 (Apache-2.0). The model card gives no training data description or cutoff date (contamination test in `06_results.ipynb`).

- The model is loaded with an explicit `device`: `TimestralForecaster.from_pretrained` defaults to CPU.
- `forecast()` is called with its default parameters except **`clip_nonnegative=False`**. The default clamps every quantile at 0 when the whole observed context is non-negative, which can happen on an 85-day window and would forbid negative prices.
- Context `(1, 1, 2048)` (batch, variate, time); covariate version: `known_covariates` of shape `(1, 4, 2072)`, the same EXOG array as TimesFM 3. `non_working_day` only takes the values 0 and 1, so Timestral treats it as an indicator.
- Timestral returns 21 quantile levels (0.01, 0.05, 0.10 … 0.90 by 0.05, 0.95, 0.99). The deciles 0.1 … 0.9 are exact levels among them (indices 2, 4, …, 18), so they are stored without interpolation. The point forecast is the median.
- Inference is deterministic (no sampling), so no seed is needed.

In [ ]:
# Free the TimesFM 3 weights so that the peak VRAM below only measures Timestral
del timesfm_model, timesfm, timesfm_cov
if device == "cuda":
    torch.cuda.empty_cache()

start = time.perf_counter()
timestral_model = Timestral.load(device)
print(f"Loaded in {time.perf_counter() - start:.1f} s on {timestral_model.device}, "
      f"max context {timestral_model.max_context}, known covariates: {timestral_model.supports_known_covariates}")
print(f"Quantile levels: {timestral_model.quantile_levels}")

timestral = Timestral(timestral_model, with_covariates=False)
timestral_cov = Timestral(timestral_model, with_covariates=True)

### 2.1 Check on one day

Same day as for TimesFM 3: context (last week shown), median and 10–90 % band of both versions, actual prices of D.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(history.index[-7 * 24:], history[cfg.TARGET].iloc[-7 * 24:], color="k", lw=1, label="context (last week)")
ax.plot(actual.index, actual, color="k", lw=1, ls="--", label="actual prices of D")
for model, label, color in [(timestral, "Timestral", "C2"), (timestral_cov, "Timestral + cov.", "C3")]:
    y_pred, q = model.predict(history, future_cov)
    ax.plot(future_cov.index, y_pred, color=color, label=f"{label} median")
    ax.fill_between(future_cov.index, q["q10"], q["q90"], color=color, alpha=0.2, label=f"{label} 10–90 %")
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("EUR/MWh")
ax.set_title(f"Timestral-209M forecasts for {day.date()}")
ax.legend(fontsize=8, ncol=3)
fig.tight_layout()
plt.show()

### 2.2 Rolling evaluation

Same loop, same days and same inputs as TimesFM 3.

In [ ]:
for name, model in [("timestral", timestral), ("timestral_cov", timestral_cov)]:
    res[name] = timed_run(model, name, cfg.TEST_DAYS)

In [ ]:
for name, model in [("timestral", timestral), ("timestral_cov", timestral_cov)]:
    res_control[name] = timed_run(model, name, CONTROL_DAYS, results_dir=cfg.RESULTS_DIR / "control")

### 2.3 First look at the results

Same summary as for TimesFM 3, now for the four foundation model runs.

In [ ]:
pd.concat({"main test": summary(res), "control": summary(res_control)}).round(3)

On the main test, the covariates reduce the MAE of both foundation models by about 40 % (TimesFM 3: 24.3 → 14.4 EUR/MWh, Timestral: 22.8 → 13.0), and Timestral is slightly ahead of TimesFM 3 in both versions. Both models are worse on the control period, but the control period is shorter (24 days) and falls in another season: whether this points to contamination can only be judged against the classical models on the same days (rMAE, `06_results.ipynb`).

With `clip_nonnegative=False`, Timestral does forecast negative prices (275 and 283 negative hourly medians on the main test, for 425 negative actual hours).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, sharey=True)
for ax, (name, label) in zip(axes, [("timestral", "Timestral"), ("timestral_cov", "Timestral + cov.")]):
    r = res[name][res[name]["target_date"].isin(week)]
    t = r["target_date"] + pd.to_timedelta(r["hour"], unit="h")
    ax.plot(t, r["y_true"], color="k", lw=1, label="actual")
    ax.plot(t, r["y_pred"], color="C2", lw=1, label="median")
    ax.fill_between(t, r["q10"], r["q90"], color="C2", alpha=0.2, label="10–90 %")
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_title(label)
    ax.set_ylabel("EUR/MWh")
    ax.legend(fontsize=8, loc="upper right")
fig.suptitle(f"Day-ahead forecasts, first week of the main test ({week[0].date()} → {week[-1].date()})")
fig.tight_layout()
plt.show()

In [ ]:
# MAE by hour of day, main test, the four foundation model runs
by_hour = pd.DataFrame({
    name: r.assign(err=(r["y_true"] - r["y_pred"]).abs()).groupby("hour")["err"].mean() for name, r in res.items()
})
ax = by_hour.plot(figsize=(10, 3.5), marker="o", ms=3)
ax.set_ylabel("MAE [EUR/MWh]")
ax.set_title("Foundation models: MAE by hour of day, main test")
plt.tight_layout()
plt.show()

### 2.4 Compute

Run of 2026-11-19 on the NVIDIA RTX 3070 (8 GB, driver 580, PyTorch 2.15), 207 days per model (main test + control), TimesFM 3 weights freed beforehand:

| Model | Time per day | Main test (183 days) | Control (24 days) | Peak VRAM |
|---|---|---|---|---|
| `timestral` | ≈ 0.05 s | 9 s | 1.2 s | 0.80 GB |
| `timestral_cov` | ≈ 0.09 s | 17 s | 2.2 s | 0.82 GB |

Model loading: ≈ 1.5 s (weights already in the Hugging Face cache). The times match the smoke test (≈ 50 ms univariate, ≈ 90 ms with covariates, ≈ 0.85 GB peak VRAM). Timestral is about 2 to 3.5 times faster than TimesFM 3 and uses less memory, although `forecast()` runs two passes per call (default `symmetric_averaging=True`: the forecast of −context is mirrored and averaged in).